# Differential Equations — Session 28
## Section 6.3: Solutions About Singular Points

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Classify regular and irregular singular points; state Frobenius' theorem; construct a Frobenius trial series; derive an indicial equation; interpret root differences; generate coefficients; and recognize when logarithms are required.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–18 min | Regular versus irregular singular points |
| 18–35 min | Frobenius theorem |
| 35–62 min | Indicial equation and recurrence |
| 62–78 min | Three root cases |
| 78–88 min | Local behavior |
| 88–90 min | Exit check |

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.special import jv, yv, iv, kv, eval_legendre, jn_zeros
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)
def polynomial_value(coefficients, x):
    x = np.asarray(x, dtype=float)
    total = np.zeros_like(x)
    for n, c in enumerate(coefficients):
        total += c*x**n
    return total
print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 6.3-A — Regular singular point

For
$$
y''+P(x)y'+Q(x)y=0,
$$
a singular point $x_0$ is regular singular when
$$
(x-x_0)P(x)
$$
and
$$
(x-x_0)^2Q(x)
$$
are analytic at $x_0$. Otherwise it is irregular singular.

### Theorem 6.3-B — Frobenius theorem

At a regular singular point, at least one solution has the form
$$
y=(x-x_0)^r\sum_{n=0}^{\infty}c_n(x-x_0)^n,
\qquad c_0\ne0.
$$

### Definition 6.3-C — Indicial equation

The coefficient of the lowest power after substitution gives a polynomial equation in $r$, called the indicial equation.

### Theorem 6.3-D — Three root cases

If $r_1\ge r_2$ are real:

1. noninteger difference: two Frobenius series;
2. positive-integer difference: the second solution may contain $\ln(x-x_0)$;
3. repeated root: a logarithmic second solution is required.

### Reduction formula

If $y_1$ is known,
$$
y_2=y_1\int\frac{e^{-\int P(x)\,dx}}{y_1(x)^2}\,dx.
$$

### Classroom Checkpoint — Regular Singular Test

For

$$
y''+P(x)y'+Q(x)y=0,
$$

what must be analytic at a regular singular point $x_0$?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Pole order and classification

In [ ]:
x = np.linspace(0.05, 2, 600)
plt.loglog(x, 1/x, label=r"$1/x$")
plt.loglog(x, 1/x**2, label=r"$1/x^2$")
plt.loglog(x, 1/x**3, label=r"$1/x^3$")
plt.xlabel("distance from singular point")
plt.ylabel("magnitude")
plt.title("Pole order near a singularity")
plt.legend(); plt.show()

For
$$
x^2y''+xy'+(x^2-4)y=0,
$$
we have $P=1/x$ and $Q=1-4/x^2$. Thus $xP$ and $x^2Q$ are analytic, so zero is regular singular.

## 2. Frobenius derivatives

At $x_0=0$,
$$
y=\sum_{n=0}^{\infty}c_nx^{n+r},
$$
$$
y'=\sum_{n=0}^{\infty}(n+r)c_nx^{n+r-1},
$$
$$
y''=\sum_{n=0}^{\infty}(n+r)(n+r-1)c_nx^{n+r-2}.
$$

## 3. Two noninteger-separated roots

For
$$
3xy''+y'-y=0,
$$
the indicial equation is
$$
r(3r-2)=0.
$$
Hence $r=0$ and $r=2/3$, with recurrence
$$
c_{k+1}=
\frac{c_k}{(k+r+1)(3k+3r+1)}.
$$

In [ ]:
def frobenius_coefficients(r, N=10):
    c = np.zeros(N+1); c[0] = 1
    for k in range(N):
        c[k+1] = c[k]/((k+r+1)*(3*k+3*r+1))
    return c

for r in [0, 2/3]:
    print("r =", r, frobenius_coefficients(r, 8))

In [ ]:
x = np.linspace(0.001, 4, 700)
for r in [0, 2/3]:
    c = frobenius_coefficients(r, 16)
    plt.plot(x, x**r*polynomial_value(c, x), label=fr"$r={r}$")
plt.legend()
plt.title(r"Frobenius families for $3xy''+y'-y=0$")
plt.show()

In [ ]:
def frobenius_explorer(root="0", N=10):
    r = 0.0 if root == "0" else 2/3
    x = np.linspace(0.001, 5, 700)
    y = x**r*polynomial_value(frobenius_coefficients(r, N), x)
    plt.plot(x, y)
    plt.title(fr"$r={r}$, truncation degree {N}")
    plt.show()
if WIDGETS_AVAILABLE:
    interact(frobenius_explorer,
             root=Dropdown(options=["0", "2/3"], value="0"),
             N=IntSlider(min=2, max=30, value=10))
else:
    frobenius_explorer()

## 4. Repeated roots and logarithms

For
$$
x^2y''-xy'+y=0,
$$
the indicial equation is $(r-1)^2=0$. The solutions are
$$
y_1=x,\qquad y_2=x\ln x
$$
on $x>0$.

In [ ]:
x = np.linspace(0.01, 4, 600)
plt.plot(x, x, label=r"$x$")
plt.plot(x, x*np.log(x), label=r"$x\ln x$")
plt.axhline(0, linestyle="--")
plt.legend()
plt.title("Repeated indicial root")
plt.show()

## 5. Positive-integer root difference

For
$$
x^2y''+xy'-y=0,
$$
the roots are $1$ and $-1$. Here both $x$ and $x^{-1}$ exist without a logarithm. In general, an integer difference requires careful recurrence analysis.

## 6. Local behavior from the exponent

In [ ]:
x = np.linspace(0.001, 3, 700)
for r in [-1, -0.5, 0, 0.5, 1]:
    plt.plot(x, x**r, label=fr"$x^{{{r}}}$")
plt.ylim(0, 10)
plt.legend()
plt.title("Indicial exponents control local behavior")
plt.show()

## Exit check

For
$$
x^3y''+xy'+y=0,
$$
$P=1/x^2$, so $xP=1/x$ is not analytic. Therefore zero is irregular singular.

## Classroom Checkpoint — Closing Reflection

Before continuing, try to state the central method or theorem of this lesson, including its assumptions and one situation in which it is useful.

> Discuss first; run the next cell for an instructor summary.